# Retrieve Chunks from Preprocessed ChromaDB

Test retrieval of chunks from the ChromaDB created by `01_llm_chunking_embedding`.

In [4]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Path setup — find preprocessed_db (works from repo root or experiments/notebooks)
_cwd = Path.cwd()
_candidates = [_cwd, _cwd.parent, _cwd.parent.parent]
PREPROCESSED_DB = None
for root in _candidates:
    p = root / "experiments" / "scripts" / "01_llm_chunking_embedding" / "preprocessed_db"
    if p.exists():
        PREPROCESSED_DB = p
        break
if PREPROCESSED_DB is None:
    PREPROCESSED_DB = _cwd / "experiments" / "scripts" / "01_llm_chunking_embedding" / "preprocessed_db"

EXPERIMENTS_PATH = PREPROCESSED_DB.parent.parent.parent  # experiments/
BACKEND_PATH = EXPERIMENTS_PATH.parent / "backend"

if str(EXPERIMENTS_PATH) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_PATH))

# Load environment variables from backend/.env if it exists;
# otherwise, load system environment variables with default behavior
env_path = BACKEND_PATH / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f"[OK] Loaded environment variables from {env_path}")
else:
    load_dotenv(override=True)
    print("[WARNING] Backend .env not found, using default environment")

COLLECTION_NAME = "docs"
EMBEDDING_MODEL = "text-embedding-3-large"

print(f"DB path: {PREPROCESSED_DB}")
print(f"Exists: {PREPROCESSED_DB.exists()}")

[OK] Loaded environment variables from c:\Users\smvan\repos\madetech-rag-assistant\backend\.env
DB path: c:\Users\smvan\repos\madetech-rag-assistant\experiments\scripts\01_llm_chunking_embedding\preprocessed_db
Exists: True


In [5]:
# Connect to ChromaDB and verify the collection
from chromadb import PersistentClient

chroma = PersistentClient(path=str(PREPROCESSED_DB))
collection = chroma.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' has {collection.count()} chunks")

Collection 'docs' has 163 chunks


In [8]:
# Test semantic search: embed query and retrieve similar chunks
from openai import OpenAI

client = OpenAI()
query = "What cycling benefits do we have?"
query_embedding = client.embeddings.create(
    model=EMBEDDING_MODEL, input=[query]
).data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

print(f"Query: {query}\n")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
), 1):
    print(f"--- Result {i} (distance={dist:.4f}) ---")
    print(f"Metadata: {meta}")
    print(f"Content preview: {doc[:200]}...")
    print()

Query: What cycling benefits do we have?

--- Result 1 (distance=1.0197) ---
Metadata: {'category': 'Benefits', 'id': 'benefits-cycle_to_work_scheme', 'title': 'Cycle To Work'}
Content preview: Headline: CycleScheme Overview
Summary: CycleScheme offers a cost‑effective salary‑sacrifice benefit, saving 25–39% on tax and NI, with a 12‑month deduction schedule.
Original Text:
### Who are CycleS...

--- Result 2 (distance=1.0260) ---
Metadata: {'category': 'Benefits', 'id': 'benefits-cycle_to_work_scheme', 'title': 'Cycle To Work'}
Content preview: Headline: Intro: Cycle to Work Overview
Summary: Made Tech supports cycling through a £2500 CycleScheme limit, salary‑sacrifice for 12 months, ownership options post‑period, FAQ links, and final salar...

--- Result 3 (distance=1.1619) ---
Metadata: {'title': 'Cycle To Work', 'category': 'Benefits', 'id': 'benefits-cycle_to_work_scheme'}
Content preview: Headline: Interest Free vs. CycleScheme
Summary: CycleScheme saves 25–39% on tax/NI and offe